<a href="https://colab.research.google.com/github/inna-tsymb/structural_bioinformatics/blob/main/HIV_WT_Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Встановлення AutoDock Vina та інструментів для конвертації
!sudo apt-get update
!sudo apt-get install -y autodock-vina openbabel

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Ge

In [2]:
# Підготовка Grid Box
import numpy as np
import os

def get_box_center(pdb_file):
    # Перевіряємо, чи фізично існує файл за цією адресою
    if not os.path.exists(pdb_file):
        print(f"❌ Помилка: Файл '{pdb_file}' не знайдено!")
        print("Переконайтеся, що ви завантажили його на панель зліва.\n")
        return

    coords = []
    with open(pdb_file, 'r') as f:
        for line in f:
            # Шукаємо рядки з координатами атомів
            if line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])

    if len(coords) == 0:
        print(f"❌ Помилка: У файлі '{pdb_file}' немає атомів. Можливо, він зберігся неправильно?\n")
        return

    coords = np.array(coords)
    center = np.mean(coords, axis=0)
    print(f"✅ Координати центру для {pdb_file}:")
    print(f"center_x = {center[0]:.3f}")
    print(f"center_y = {center[1]:.3f}")
    print(f"center_z = {center[2]:.3f}\n")

# У Colab файли лежать у папці /content/
get_box_center('/content/DRV_ligand.pdb')

✅ Координати центру для /content/DRV_ligand.pdb:
center_x = 8.280
center_y = -16.385
center_z = 0.027



In [3]:
# Конвертація файлів у формат .pdbqt
!obabel PR_WT_receptor.pdb -O PR_WT_receptor.pdbqt -p 7.4 -xr
!obabel DRV_ligand.pdb -O DRV_ligand.pdbqt -p 7.4

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is PR_WT_receptor.pdb)

1 molecule converted
1 molecule converted


In [4]:
# Створення конфігураційного файлу для докінгу Vina
config_pr = """receptor = PR_WT_receptor.pdbqt
ligand = DRV_ligand.pdbqt

center_x = 8.280
center_y = -16.385
center_z = 0.027

size_x = 25
size_y = 25
size_z = 25

exhaustiveness = 10
"""

with open("config_WT_PR.txt", "w") as f:
    f.write(config_pr)

print("✅ Конфігураційний файл успішно оновлено та створено!")

✅ Конфігураційний файл успішно оновлено та створено!


In [5]:
# Запуск докінгу
!echo "=== ДОКІНГ ПРОТЕАЗИ (PR) + ДАРУНАВІР ==="
!vina --config config_WT_PR.txt --out PR_WT_results.pdbqt

=== ДОКІНГ ПРОТЕАЗИ (PR) + ДАРУНАВІР ===
AutoDock Vina v1.2.3
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see h

In [6]:
# Найкращі пози для молекулярної динаміки
!echo "Крок 1: Витягуємо найкращі пози лігандів (Mode 1)..."
!obabel -ipdbqt PR_WT_results.pdbqt -f 1 -l 1 -opdb -O PR_ligand_best.pdb

!echo "Крок 2: Склеюємо рецептори з лігандами у єдині комплекси..."
# Для Протеази
!grep -v "^END" PR_WT_receptor.pdb > PR_WT_complex.pdb
!grep "^ATOM\|^HETATM" PR_ligand_best.pdb >> PR_WT_complex.pdb
!echo "END" >> PR_WT_complex.pdb

# Створюємо чисті ліганди для молекулярної динаміки
!grep -v "^MODEL" PR_ligand_best.pdb | grep -v "^ENDMDL" > PR_clean_ligand.pdb

!echo "✅ Готово! Файли PR_WT_complex.pdb та PR_clean_ligand.pdb успішно створені."

Крок 1: Витягуємо найкращі пози лігандів (Mode 1)...
1 molecule converted
Крок 2: Склеюємо рецептори з лігандами у єдині комплекси...
✅ Готово! Файли PR_WT_complex.pdb та PR_clean_ligand.pdb успішно створені.
